[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/ab-test-practice/blob/main/assignments/week01_assignment_designing_ab_tests.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 1 Assignment: Designing A/B Tests

## FamilyNest - Growing Homes on the Platform

In this assignment, you will practice designing and analyzing A/B tests in a realistic product setting.

---
## Setup

Run the cell below to import the required libraries and helper functions.

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats

def get_ci(mean_treatment, mean_control, n_treatment, n_control, ci=0.95):
    sd = ((mean_treatment * (1 - mean_treatment)) / n_treatment + (mean_control * (1 - mean_control)) / n_control)**0.5
    lift = mean_treatment - mean_control
    val = stats.norm.isf((1 - ci) / 2)
    lwr_bnd = lift - val * sd
    upr_bnd = lift + val * sd
    return (lwr_bnd, upr_bnd)

def calculate_results(df, metric):
    if metric not in ('new_active_listing', 'new_booked_listing', 'new_cancelled_listing'):
        raise Exception("Invalid metric")
    mean_control = df.loc[df['variant'] == "control", metric].mean()
    mean_treatment = df.loc[df['variant'] == "treatment", metric].mean()
    abs_diff = mean_treatment - mean_control
    rel_diff = (mean_treatment - mean_control) / mean_control
    data_group1 = list(df.query('variant == "control"')[metric])
    data_group2 = list(df.query('variant == "treatment"')[metric])
    results = stats.ttest_ind(a=data_group1, b=data_group2, equal_var=True)
    [ci_low, ci_high] = get_ci(mean_treatment, mean_control, len(data_group2), len(data_group1), .95)
    ci_low = ci_low / mean_control
    ci_high = ci_high / mean_control
    print("rel_diff, abs_diff, pvalue, ci_low, ci_high")
    return (rel_diff, abs_diff, results.pvalue, ci_low, ci_high)

---
## Background

**FamilyNest** is a platform similar to Airbnb, specifically designed for families with young kids. You are a **product data scientist** on the team focused on **growing homes on the platform** by improving the host onboarding flow.

Your product manager, **PaM**, wants to run experiments to improve the onboarding experience for new hosts. Before designing any experiments, let's understand the current state of the host onboarding funnel.

### Host Onboarding Funnel

| User journey step | Conversion from last step | Conversion from start | Median time since last step |
|---|---|---|---|
| Start onboarding flow | N/A | N/A | N/A |
| Publish home (finish onboarding) | 40% | 40% | 2 hours |
| Home is booked by a guest | 50% | 20% | 7 days |
| First guest stays | 90% | 18% | 14 days |
| Home receives first review | 60% | 10.8% | 2 days |

Keep this funnel in mind as you work through the tasks below. The conversion rates and timing will inform your decisions about metric selection and test duration.

---
## Task 1: Identify Target and Guardrail Metrics

### Task 1.a. Pros and Cons of Target Metrics

We are considering three possible **target metrics** for our onboarding experiments. For each metric, list the pros and cons considering:
- **Sensitivity**: How easily can we detect a change in this metric?
- **Timeliness**: How quickly after the intervention does this metric respond?
- **Business connection**: How directly does this metric relate to business value?

The candidate metrics are:

1. **New Active Listings (NAL)** — A home is published (host finishes onboarding)
2. **New Booked Listings (NBL)** — A listing receives its first booking from a guest
3. **New Reviewed Listings (NRL)** — A listing receives its first review after a guest stay

<details>
<summary>💡 Hint (click to expand)</summary>

Think about the funnel table above. Metrics higher in the funnel tend to be more sensitive and timely (more events, shorter delay), but may be less connected to actual business value. Metrics lower in the funnel are closer to revenue but take longer to observe and have fewer events (lower statistical power).

</details>

**Your answer:**

| Metric | Pros | Cons |
|--------|------|------|
| NAL | | |
| NBL | | |
| NRL | | |

### Task 1.b. Choose a Target Metric

Based on your analysis above, which metric should we use as the **primary target metric** for our onboarding experiments? Justify your choice.

<details>
<summary>💡 Hint (click to expand)</summary>

There is no single "correct" answer, but consider which metric best balances all three dimensions. A good target metric should be sensitive enough to detect changes within a reasonable test duration, while still being meaningfully connected to business outcomes.

</details>

**Your answer:**



### Task 1.c. Guardrail Metrics

What **guardrail metrics** should we track alongside our target metric? Guardrail metrics help us ensure we're not causing unintended harm while optimizing for the target.

<details>
<summary>💡 Hint (click to expand)</summary>

Think about what could go wrong. For example, if we make it too easy to publish a listing, we might get low-quality listings that lead to cancellations. Also consider the guest side of the marketplace — are there metrics that would indicate guest experience is suffering?

</details>

**Your answer:**



---
## Task 2: Design and Analyze — Auto-generate Title Feature

The team wants to test **auto-generating a title** for the listing based on the address and number of rooms. Currently, hosts must write their own title during onboarding. The hypothesis is that removing this friction will help more hosts complete the onboarding flow and ultimately get booked.

### Task 2.a. Test Design

**2.a.i. Triggering and Eligibility**

When should the test be triggered? Who is eligible to be in this experiment?

<details>
<summary>💡 Hint (click to expand)</summary>

Think about at what point in the user journey this feature becomes relevant. Should existing hosts who are creating a second listing be included? Should the randomization happen at the start of onboarding or at the title step?

</details>

**Your answer:**



**2.a.ii. Hypothesis**

Write a clear hypothesis for this experiment.

<details>
<summary>💡 Hint (click to expand)</summary>

A good hypothesis follows the format: "If we [change], then [metric] will [direction] because [reasoning]."

</details>

**Your answer:**



**2.a.iii. Sample Size and Duration**

How long should the test run? Use the following assumptions:
- **Traffic**: 5,000 new users per day enter the onboarding flow
- **Target metric**: NBL (New Booked Listings)
- **Baseline rate**: ~20% (from the funnel table)
- **Minimum Detectable Effect (MDE)**: 5% relative lift (i.e., moving from 20% to 21%)
- **Power**: 80%
- **Significance level**: 5% (two-sided)

Calculate the minimum sample size per variant, then determine how many days the test needs to run.

<details>
<summary>💡 Hint (click to expand)</summary>

For a two-proportion z-test, the sample size per group can be calculated as:

```
n = (Z_alpha/2 + Z_beta)^2 * (p1*(1-p1) + p2*(1-p2)) / (p2 - p1)^2
```

Where:
- Z_alpha/2 = 1.96 (for 5% significance, two-sided)
- Z_beta = 0.84 (for 80% power)
- p1 = baseline rate = 0.20
- p2 = baseline * (1 + relative_lift) = 0.20 * 1.05 = 0.21

Remember: with 5,000 users/day split into 2 variants, each variant gets 2,500 users/day.

</details>

In [ ]:
# Your code here
# Calculate the minimum sample size per variant and the test duration in days

# Parameters
# baseline_rate = 
# mde_relative = 
# alpha = 
# power = 
# daily_traffic = 

# Calculate sample size per variant

# Calculate duration


**Your answer:**

- Minimum sample size per variant: 
- Test duration: 


### Task 2.b. Test Analysis

The experiment has been run! Load the data and analyze the results.

In [ ]:
df_autotitle = pd.read_csv('../data/dataset_autotitle.csv')
df_autotitle.head()

In [ ]:
# Your code here
# Explore the dataset: check shape, columns, value counts for variant, etc.


Now answer the following questions:

1. What is the **relative lift** and **absolute lift** in NBL (New Booked Listings)?
2. Is the result **statistically significant**?
3. What about the **guardrail metric** (New Cancelled Listings - NCL)? Is it moving in a concerning direction?
4. Should we **launch** this feature? Write a brief summary/recommendation for PaM.

<details>
<summary>💡 Hint (click to expand)</summary>

Use the `calculate_results()` helper function defined in the Setup section. Call it with the dataframe and the metric name (e.g., `'new_booked_listing'` or `'new_cancelled_listing'`).

</details>

In [ ]:
# Your code here
# Analyze the target metric: NBL (new_booked_listing)


In [ ]:
# Your code here
# Analyze the guardrail metric: NCL (new_cancelled_listing)


**Your answer / Recommendation for PaM:**



---
## Task 3: Design and Analyze — Help Chat Feature

The team wants to test adding a **chat button** that gives hosts access to live support during onboarding. Unlike the auto-title feature which only affects the onboarding flow, this chat button targets the **entire help center**, so the eligible traffic is much higher.

- **Traffic**: 80,000 users/day visit the help center
- **Target metric**: NBL (baseline ~20%)
- **MDE**: 2% relative lift
- **Power**: 80%
- **Significance level**: 5% (two-sided)

### Task 3.a. Test Duration

How long should this test run? Calculate the minimum sample size per variant and determine the duration.

<details>
<summary>💡 Hint (click to expand)</summary>

Same formula as Task 2.a.iii, but with different MDE (2% relative lift) and different daily traffic (80,000 users/day split into 2 variants = 40,000/variant/day).

</details>

In [ ]:
# Your code here
# Calculate sample size and duration for the help chat experiment


**Your answer:**

- Minimum sample size per variant: 
- Test duration: 


### Task 3.b. Test Analysis

The experiment has been run. Load the data and analyze the results. Make a launch recommendation for PaM.

In [ ]:
df_help_center = pd.read_csv('../data/dataset_help_center.csv')
df_help_center.head()

In [ ]:
# Your code here
# Explore the dataset


In [ ]:
# Your code here
# Analyze the target metric: NBL (new_booked_listing)


In [ ]:
# Your code here
# Analyze the guardrail metric: NCL (new_cancelled_listing)


**Your answer / Recommendation for PaM:**



---
## Bonus 1: SUTVA

**SUTVA (Stable Unit Treatment Value Assumption)** states that the treatment applied to one unit does not affect the outcome of another unit. This is a fundamental assumption in A/B testing.

How might A/B testing in the onboarding flow **violate SUTVA**? Think about the FamilyNest marketplace context and how hosts and guests interact.

<details>
<summary>💡 Hint (click to expand)</summary>

Consider: if the treatment causes more hosts to publish listings, does that affect the booking probability of other hosts (both in treatment and control)? Think about supply and demand dynamics in a marketplace.

</details>

**Your answer:**



---
## Bonus 2: Text-to-Clicker Feature

The team wants to convert a **free-text description step** in the onboarding flow to a **clicker/checkbox interface**. Instead of writing a paragraph about their home's amenities, hosts would select from a pre-defined list of options.

- **Traffic**: 5,000 users/day enter the onboarding flow
- **Target metric**: NBL (baseline ~20%)
- PaM doesn't have a strong preference for MDE — use your judgment based on what would be a meaningful and detectable change.

**Your tasks:**
1. Choose an appropriate MDE and justify your choice
2. Calculate the test duration
3. Load the data and analyze the results
4. Make a launch recommendation

<details>
<summary>💡 Hint (click to expand)</summary>

When choosing an MDE, consider: What is the smallest effect that would be worth launching? Think about engineering cost, user disruption, and business impact. A common approach is to pick the smallest effect that would justify the effort of building and maintaining the feature.

</details>

In [ ]:
# Your code here
# Choose an MDE, calculate sample size and test duration


In [ ]:
df_text_to_clicker = pd.read_csv('../data/dataset_text_to_clicker.csv')
df_text_to_clicker.head()

In [ ]:
# Your code here
# Explore the dataset


In [ ]:
# Your code here
# Analyze the target metric: NBL (new_booked_listing)


In [ ]:
# Your code here
# Analyze the guardrail metric: NCL (new_cancelled_listing)


**Your answer / Recommendation for PaM:**

- Chosen MDE and justification:
- Test duration:
- Analysis results:
- Launch recommendation:
